# 01 — Tensor Fundamentals (Sections 1 & 2)

This notebook covers:
- **Section 1 (coding):** Check whether CUDA is available; identify the available computation device; compare basic properties of a NumPy array and a PyTorch tensor.
- **Section 2 (coding):** Create scalar/vector/matrix/3D tensors; create zeros/ones tensors; generate uniform/normal random tensors; create sequences with range operations; inspect shape/rank/dtype/device; experiment with float/int dtypes; create tensors with different shapes and explain dimensions.

Study topics supported: PyTorch vs NumPy, tensors vs arrays, CPU vs GPU, eager execution, tensor-based computation, PyTorch's role in DL; scalar/vector/matrix/higher-D tensors, rank, shape, dimensions, dtypes, random generation, initialization.


In [1]:
import numpy as np
import torch

print('NumPy version:', np.__version__)
print('PyTorch version:', torch.__version__)


NumPy version: 2.5.4
PyTorch version: 2.15.0+cpu


## Section 1 — Coding Exercises
### 1. Check whether CUDA is available

In [2]:
cuda_available = torch.cuda.is_available()
print('CUDA available:', cuda_available)
if cuda_available:
  print('CUDA device count:', torch.cuda.device_count())
  print('Current CUDA device index:', torch.cuda.current_device())
  print('CUDA device name:', torch.cuda.get_device_name(0))
else:
  print('No CUDA device detected. Using CPU/MPS fallback as available.')


CUDA available: False
No CUDA device detected. Using CPU/MPS fallback as available.


### 2. Identify the available computation device

Device selection pattern: **CUDA -> MPS -> CPU** (consistent across notebooks).


In [3]:
def get_device():
  '''Return the preferred computation device.'''
  if torch.cuda.is_available():
    return torch.device('cuda')
  if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    return torch.device('mps')
  return torch.device('cpu')

device = get_device()
print('Selected device:', device)
sample = torch.randn(2, 3, device=device)
print('Sample tensor device:', sample.device)
print(sample)
sample_cpu = sample.detach().to('cpu')
print('CPU copy for NumPy interop:', tuple(sample_cpu.shape), sample_cpu.device)


Selected device: cpu
Sample tensor device: cpu
tensor([[ 0.3899, -0.9132, -0.7666],
        [-0.2808, -1.6509,  0.2398]])
CPU copy for NumPy interop: (2, 3) cpu


### 3. Compare the basic properties of a NumPy array and a PyTorch tensor

Comparison includes type, shape, ndim/rank, dtype, device semantics, and gradient tracking (`requires_grad` is N/A for NumPy arrays).


In [4]:
np_arr = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
torch_cpu = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
torch_dev = torch_cpu.to(device)

print('NumPy array:')
print(np_arr)
print('  type:', type(np_arr))
print('  shape:', np_arr.shape)
print('  ndim:', np_arr.ndim)
print('  dtype:', np_arr.dtype)
print('  device: N/A (NumPy arrays are CPU-resident; no torch.device attribute)')
print('  requires_grad: N/A (NumPy arrays do not track autograd)')

print('\nPyTorch tensor (CPU):')
print(torch_cpu)
print('  type:', type(torch_cpu))
print('  shape:', tuple(torch_cpu.shape))
print('  ndim:', torch_cpu.ndim)
print('  dtype:', torch_cpu.dtype)
print('  device:', torch_cpu.device)
print('  requires_grad:', torch_cpu.requires_grad)

print('\nPyTorch tensor (selected device):')
print('  device:', torch_dev.device)
print('  requires_grad:', torch_dev.requires_grad)
print('  values equal to CPU tensor:', torch.equal(torch_cpu, torch_dev.detach().to('cpu')))

np_from_torch = torch_dev.detach().to('cpu').numpy()
print('\nNumPy conversion from tensor (via CPU):', np_from_torch, np_from_torch.dtype)


NumPy array:
[[1. 2. 3.]
 [4. 5. 6.]]
  type: <class 'numpy.ndarray'>
  shape: (2, 3)
  ndim: 2
  dtype: float32
  device: N/A (NumPy arrays are CPU-resident; no torch.device attribute)
  requires_grad: N/A (NumPy arrays do not track autograd)

PyTorch tensor (CPU):
tensor([[1., 2., 3.],
        [4., 5., 6.]])
  type: <class 'torch.Tensor'>
  shape: (2, 3)
  ndim: 2
  dtype: torch.float32
  device: cpu
  requires_grad: False

PyTorch tensor (selected device):
  device: cpu
  requires_grad: False
  values equal to CPU tensor: True

NumPy conversion from tensor (via CPU): [[1. 2. 3.]
 [4. 5. 6.]] float32


## Section 2 — Coding Exercises
Study topics supported: scalar/vector/matrix/higher-dimensional tensors, rank, shape, dimensions, dtypes, random generation, initialization.


### 1. Create scalar, vector, matrix, and 3D tensors

In [5]:
scalar = torch.tensor(7)
vector = torch.tensor([1, 2, 3, 4])
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
tensor_3d = torch.arange(24).reshape(2, 3, 4)

print('Scalar:', scalar, 'shape:', tuple(scalar.shape), 'rank:', scalar.ndim)
print('Vector:', vector, 'shape:', tuple(vector.shape), 'rank:', vector.ndim)
print('Matrix:\n', matrix)
print('shape:', tuple(matrix.shape), 'rank:', matrix.ndim)
print('3D tensor:\n', tensor_3d)
print('shape:', tuple(tensor_3d.shape), 'rank:', tensor_3d.ndim)


Scalar: tensor(7) shape: () rank: 0
Vector: tensor([1, 2, 3, 4]) shape: (4,) rank: 1
Matrix:
 tensor([[1, 2, 3],
        [4, 5, 6]])
shape: (2, 3) rank: 2
3D tensor:
 tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
shape: (2, 3, 4) rank: 3


### 2. Create tensors filled with zeros and ones

In [6]:
zeros_2x3 = torch.zeros(2, 3)
ones_3x2 = torch.ones(3, 2)
zeros_like_matrix = torch.zeros_like(matrix)
ones_like_matrix = torch.ones_like(matrix)
print('zeros(2,3):\n', zeros_2x3)
print('ones(3,2):\n', ones_3x2)
print('zeros_like(matrix):\n', zeros_like_matrix)
print('ones_like(matrix):\n', ones_like_matrix)


zeros(2,3):
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones(3,2):
 tensor([[1., 1.],
        [1., 1.],
        [1., 1.]])
zeros_like(matrix):
 tensor([[0, 0, 0],
        [0, 0, 0]])
ones_like(matrix):
 tensor([[1, 1, 1],
        [1, 1, 1]])


### 3. Generate random tensors using uniform and normal distributions

In [7]:
torch.manual_seed(42)
uniform_tensor = torch.rand(2, 4)
normal_tensor = torch.randn(2, 4)
randint_tensor = torch.randint(0, 10, (2, 4))
print('Uniform [0,1):\n', uniform_tensor)
print('Normal (mean~0,std~1):\n', normal_tensor)
print('Random integers [0,10):\n', randint_tensor)
print('Uniform min/max:', uniform_tensor.min().item(), uniform_tensor.max().item())
print('Normal mean/std:', normal_tensor.mean().item(), normal_tensor.std(unbiased=False).item())


Uniform [0,1):
 tensor([[0.8823, 0.9150, 0.3829, 0.9593],
        [0.3904, 0.6009, 0.2566, 0.7936]])
Normal (mean~0,std~1):
 tensor([[-1.1229, -0.1863,  2.2082, -0.6380],
        [ 0.4617,  0.2674,  0.5349,  0.8094]])
Random integers [0,10):
 tensor([[9, 3, 1, 9],
        [7, 9, 2, 0]])
Uniform min/max: 0.2565724849700928 0.9593056440353394
Normal mean/std: 0.29178619384765625 0.9436227083206177


### 4. Create a sequence of values using a range operation

In [8]:
seq_arange = torch.arange(0, 10, 2)
seq_arange_float = torch.arange(0.0, 1.0, 0.2)
seq_linspace = torch.linspace(0, 1, steps=5)
print('arange(0,10,2):', seq_arange)
print('arange(0.0,1.0,0.2):', seq_arange_float)
print('linspace(0,1,steps=5):', seq_linspace)


arange(0,10,2): tensor([0, 2, 4, 6, 8])
arange(0.0,1.0,0.2): tensor([0.0000, 0.2000, 0.4000, 0.6000, 0.8000])
linspace(0,1,steps=5): tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


### 5. Inspect the shape, rank, data type, and device of each tensor

In [9]:
tensors = {
  'scalar': scalar, 'vector': vector, 'matrix': matrix, 'tensor_3d': tensor_3d,
  'zeros_2x3': zeros_2x3, 'uniform_tensor': uniform_tensor, 'seq_arange': seq_arange,
}
for name, t in tensors.items():
  print('---', name, '---')
  print(t)
  print('shape:', tuple(t.shape), 'rank:', t.ndim, 'dtype:', t.dtype, 'device:', t.device)
  print()


--- scalar ---
tensor(7)
shape: () rank: 0 dtype: torch.int64 device: cpu

--- vector ---
tensor([1, 2, 3, 4])
shape: (4,) rank: 1 dtype: torch.int64 device: cpu

--- matrix ---
tensor([[1, 2, 3],
        [4, 5, 6]])
shape: (2, 3) rank: 2 dtype: torch.int64 device: cpu

--- tensor_3d ---
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
shape: (2, 3, 4) rank: 3 dtype: torch.int64 device: cpu

--- zeros_2x3 ---
tensor([[0., 0., 0.],
        [0., 0., 0.]])
shape: (2, 3) rank: 2 dtype: torch.float32 device: cpu

--- uniform_tensor ---
tensor([[0.8823, 0.9150, 0.3829, 0.9593],
        [0.3904, 0.6009, 0.2566, 0.7936]])
shape: (2, 4) rank: 2 dtype: torch.float32 device: cpu

--- seq_arange ---
tensor([0, 2, 4, 6, 8])
shape: (5,) rank: 1 dtype: torch.int64 device: cpu



### 6. Experiment with floating-point and integer data types

In [10]:
int32_t = torch.tensor([1, 2, 3], dtype=torch.int32)
int64_t = torch.tensor([1, 2, 3], dtype=torch.int64)
float32_t = torch.tensor([1, 2, 3], dtype=torch.float32)
float64_t = torch.tensor([1, 2, 3], dtype=torch.float64)
bool_t = torch.tensor([1, 0, 3]) > 1
print('int32:', int32_t, int32_t.dtype)
print('int64:', int64_t, int64_t.dtype)
print('float32:', float32_t, float32_t.dtype)
print('float64:', float64_t, float64_t.dtype)
print('bool:', bool_t, bool_t.dtype)
casted = int64_t.to(torch.float32)
print('cast int64->float32:', casted, casted.dtype)
mixed = int32_t + float32_t
print('int32 + float32:', mixed, mixed.dtype)
a = torch.tensor([5, 7], dtype=torch.int32)
b = torch.tensor([2, 2], dtype=torch.int32)
print('a/b true division:', a / b, (a / b).dtype)
print('floor division:', torch.div(a, b, rounding_mode='floor'))


int32: tensor([1, 2, 3], dtype=torch.int32) torch.int32
int64: tensor([1, 2, 3]) torch.int64
float32: tensor([1., 2., 3.]) torch.float32
float64: tensor([1., 2., 3.], dtype=torch.float64) torch.float64
bool: tensor([False, False,  True]) torch.bool
cast int64->float32: tensor([1., 2., 3.]) torch.float32
int32 + float32: tensor([2., 4., 6.]) torch.float32
a/b true division: tensor([2.5000, 3.5000]) torch.float32
floor division: tensor([2, 3], dtype=torch.int32)


### 7. Create tensors with different shapes and explain what each dimension represents

In [11]:
feature_vector = torch.randn(8)
tabular_batch = torch.randn(4, 8)
grayscale_batch = torch.randn(2, 28, 28)
image_batch_nchw = torch.randn(2, 3, 32, 32)
sequence_batch = torch.randn(5, 10, 16)
examples = [
  ('feature_vector', feature_vector, '(features)', 'Single sample feature vector'),
  ('tabular_batch', tabular_batch, '(batch, features)', 'Batch of tabular records'),
  ('grayscale_batch', grayscale_batch, '(batch, height, width)', 'Batch of grayscale images'),
  ('image_batch_nchw', image_batch_nchw, '(batch, channels, height, width)', 'Batch of RGB images (NCHW)'),
  ('sequence_batch', sequence_batch, '(batch, seq_len, embed_dim)', 'Batch of token/step sequences'),
]
for name, t, axes, meaning in examples:
  print('{:16s} shape={!s:22s} axes={!s:32s} {}'.format(name, tuple(t.shape), axes, meaning))
  if name in ['feature_vector', 'tabular_batch']:
    print('  example tensor:\n', t)
  print()


feature_vector   shape=(8,)                   axes=(features)                       Single sample feature vector
  example tensor:
 tensor([ 1.3221,  0.8172, -0.7658, -0.7506,  1.3525,  0.6863, -0.3278,  0.7950])

tabular_batch    shape=(4, 8)                 axes=(batch, features)                Batch of tabular records
  example tensor:
 tensor([[-0.9138, -0.6581,  0.0780,  0.5258, -0.4880,  1.1914, -0.8140, -0.7360],
        [-1.4032,  0.0360, -0.0635,  0.6756, -0.0978,  1.8446, -1.1845,  1.3835],
        [ 1.4451,  0.8564,  2.2181,  0.5232,  0.3466, -0.1973, -1.0546,  1.2780],
        [-0.1722,  0.5238,  0.0566,  0.4263,  0.5750, -0.6417, -2.2064, -0.7508]])

grayscale_batch  shape=(2, 28, 28)            axes=(batch, height, width)           Batch of grayscale images

image_batch_nchw shape=(2, 3, 32, 32)         axes=(batch, channels, height, width) Batch of RGB images (NCHW)

sequence_batch   shape=(5, 10, 16)            axes=(batch, seq_len, embed_dim)      Batch of token/step s

## Summary
- Section 1: CUDA availability, device identification (CUDA->MPS->CPU), NumPy vs PyTorch property comparison.
- Section 2: tensor creation (scalar/vector/matrix/3D), zeros/ones, random/uniform/normal, ranges, inspection, dtypes, and shape semantics.
